In [11]:
import json
import numpy as np
import plotly.graph_objects as go
from utils import *

In [12]:
def visualize_cameras(extrinsic_matrices, Title="", scale=200.0):
    fig = go.Figure()

    # Lists for frustum lines
    all_x, all_y, all_z = [], [], []
    
    # Lists for camera centers (spheres) and labels
    cam_centers = []
    cam_labels = []

    for i in range(len(extrinsic_matrices)):
        E = extrinsic_matrices[i]
        
        # Camera-to-World matrix
        C_to_W = np.linalg.inv(E)
        cam_pos = C_to_W[:3, 3]
        cam_centers.append(cam_pos)
        cam_labels.append(f"Cam {i}")
        
        # --- Frustum Calculation ---
        s = scale
        # Z is forward in camera space
        corners_cam = np.array([
            [0, 0, 0],          # 0: Center
            [-s, -s, s*1.5],    # 1: Top-Left
            [s, -s, s*1.5],     # 2: Top-Right
            [s, s, s*1.5],      # 3: Bottom-Right
            [-s, s, s*1.5],     # 4: Bottom-Left
        ])
        
        corners_homo = np.hstack([corners_cam, np.ones((5, 1))])
        p = (C_to_W @ corners_homo.T).T[:, :3]
        
        # Line sequence for a pyramid
        indices = [0, 1, 2, 0, 2, 3, 0, 3, 4, 0, 4, 1, 1, 2, 3, 4, 1]
        for idx in indices:
            all_x.append(p[idx, 0]); all_y.append(p[idx, 1]); all_z.append(p[idx, 2])
        all_x.append(None); all_y.append(None); all_z.append(None)

    # 1. Add Frustums
    fig.add_trace(go.Scatter3d(
        x=all_x, y=all_y, z=all_z,
        mode='lines',
        line=dict(color='rgba(100, 100, 255, 0.6)', width=2),
        name='Frustums',
        hoverinfo='none'
    ))

    # 2. Add Spheres (Markers) and Labels
    cam_centers = np.array(cam_centers)
    fig.add_trace(go.Scatter3d(
        x=cam_centers[:, 0],
        y=cam_centers[:, 1],
        z=cam_centers[:, 2],
        mode='markers+text',
        marker=dict(
            size=6,
            color=np.arange(len(extrinsic_matrices)), # Color by index
            colorscale='Viridis',
            opacity=0.8
        ),
        text=cam_labels,
        textposition="top center",
        name='Camera Positions'
    ))

    # 3. Add World Origin for context
    fig.add_trace(go.Scatter3d(
        x=[0], y=[0], z=[0],
        mode='markers',
        marker=dict(size=8, color='red', symbol='cross'),
        name='World Origin'
    ))

    fig.update_layout(
        scene=dict(
            xaxis_title='X (mm)',
            yaxis_title='Y (mm)',
            zaxis_title='Z (mm)',
            aspectmode='data'
        ),
        margin=dict(l=0, r=0, b=0, t=40),
        title=f"{Title}"
    )
    
    fig.show()

In [ ]:
R_correct = np.array([
    [-1,  0,  0, 0], # Everything appear mirrored so I fix it using this
    [0,  0,  1, 0], # Swap Y and Z
    [0,  1,  0, 0],
    [0,  0,  0, 1]
])

camera_parameters_raw = read_calibration_raw('calibration_raw.json')
extrinsic_matrices_raw = extract_extrinsics_raw(camera_parameters_raw)                      # Shape (n, 4, 4) <class 'numpy.ndarray'> # Vertical up direction is "+y
extrinsic_matrices_raw = m_to_mm(extrinsic_matrices_raw)                                    # Shape (n, 4, 4) <class 'numpy.ndarray'> # Vertical up direction is "+y"
extrinsic_matrices_raw = extrinsic_matrices_raw @ R_correct
camera_matrices_raw = extract_camera_matrices_raw(camera_parameters_raw)                    # Shape (n, 3, 3) <class 'numpy.ndarray'>
distortion_coefficients_raw = extract_distortion_coefficients_raw(camera_parameters_raw)    # Shape (n, 5) <class 'numpy.ndarray'>

visualize_cameras(extrinsic_matrices_raw, "Raw VCI Data")
save_calibration_panoptic('calibration.json', extrinsic_matrices_raw, camera_matrices_raw, distortion_coefficients_raw)

Successfully saved calibration to calibration.json


In [14]:
cameras_parameters_panoptic = read_calibration_panoptic('calibration.json')      # vertical up direction is "+z"
extrinsic_matrices_panoptic = extract_extrinsics_panoptic(cameras_parameters_panoptic)   # Shape (n, 4, 4) <class 'numpy.ndarray'>
camera_matrices_panoptic = extract_camera_matrices_panoptic(cameras_parameters_panoptic) # Shape (n, 3, 3) <class 'numpy.ndarray'>
distortion_coefficients_raw_panoptic = extract_distortion_coefficients_panoptic(cameras_parameters_panoptic) # Shape (n, 5) <class 'numpy.ndarray'>

visualize_cameras(extrinsic_matrices_panoptic, "VCI Data Converted to Panoptic Convention")

In [15]:
cameras_parameters_example = read_calibration_panoptic('calibration_example.json')      # vertical up direction is "+z"
extrinsic_matrices_example = extract_extrinsics_panoptic(cameras_parameters_example)   # Shape (n, 4, 4) <class 'numpy.ndarray'>

visualize_cameras(extrinsic_matrices_example, "Example demo cameras that are provided")